# 🌿 ForestAI — EfficientNetB0 Training

**Goal:** Train a fruit identification model on 26 classes (16 edible + 10 toxic)  
**Output:** `model.tflite` + `labels.txt` saved to your Google Drive

## Before Running
1. Set Runtime to GPU → `Runtime → Change runtime type → T4 GPU → Save`
2. Upload `dataset_clean.zip` to your **Google Drive root** (not a subfolder)
3. Then run cells **one by one in order**

## Cell 1 — Check GPU

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
for g in gpus:
    print(' ', g)
    tf.config.experimental.set_memory_growth(g, True)

if not gpus:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')
else:
    print('✅ GPU ready!')

## Cell 2 — Mount Google Drive
This keeps your dataset and trained model **safe even if Colab disconnects**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive/MyDrive/')

## Cell 3 — Upload dataset_clean.zip to Drive

**Two options — choose one:**

**Option A (Recommended):** Upload manually
1. Go to [drive.google.com](https://drive.google.com)
2. Drag and drop `dataset_clean.zip` into **My Drive** (root)
3. Wait for upload to finish
4. Skip to Cell 4

**Option B:** Upload directly from this cell (slow for large files)

In [ ]:
import os
from pathlib import Path

# Check if zip already exists in Drive (Option A)
DRIVE_ROOT  = Path('/content/drive/MyDrive')
ZIP_IN_DRIVE = DRIVE_ROOT / 'dataset_clean.zip'

if ZIP_IN_DRIVE.exists():
    print(f'✅ Found dataset_clean.zip in Drive ({ZIP_IN_DRIVE.stat().st_size / 1024 / 1024:.1f} MB)')
    print('Skipping upload — proceed to Cell 4')
else:
    print('❌ dataset_clean.zip NOT found in Drive root.')
    print('Please upload it to Google Drive first, then re-run this cell.')
    print()
    print('OR run the cell below to upload directly:')

In [ ]:
# Option B — Upload directly (only run if Option A above failed)
# Comment this cell out if you used Option A

from google.colab import files
print('Select your dataset_clean.zip file...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Save to Drive so it persists
import shutil
shutil.copy(zip_name, str(DRIVE_ROOT / zip_name))
print(f'✅ Saved {zip_name} to Google Drive')

## Cell 4 — Unzip Dataset to Colab (fast local storage)

In [ ]:
import zipfile, subprocess
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
ZIP_PATH     = DRIVE_ROOT / 'dataset_clean.zip'
EXTRACT_PATH = Path('/content')
DATASET_PATH = EXTRACT_PATH / 'dataset_clean'

if DATASET_PATH.exists():
    print('Dataset already extracted. Skipping.')
else:
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print('Extraction complete ✅')

# Verify
edable_classes = list((DATASET_PATH / 'edable').iterdir()) if (DATASET_PATH / 'edable').exists() else []
toxic_classes  = list((DATASET_PATH / 'toxic').iterdir())  if (DATASET_PATH / 'toxic').exists()  else []
print(f'Edible classes : {len(edable_classes)}')
print(f'Toxic  classes : {len(toxic_classes)}')
total_imgs = len(list(DATASET_PATH.rglob('*.jpg')))
print(f'Total images   : {total_imgs}')
print('Dataset ready ✅')

## Cell 5 — Create Output Folder in Drive
All model files, checkpoints and plots will be saved here — safe from disconnects.

In [ ]:
# Output goes to Google Drive — persists after session ends
OUTPUT_DIR = DRIVE_ROOT / 'ForestAI_output'
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Output folder: {OUTPUT_DIR}')
print('All model files will be saved to Google Drive ✅')

## Cell 6 — Install Dependencies

In [ ]:
!pip install -q scikit-learn matplotlib pillow
print('Dependencies ready ✅')

## Cell 7 — Configuration

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

DATASET_ROOT    = Path('/content/dataset_clean')
# OUTPUT_DIR already set in Cell 5 (Google Drive)

IMG_SIZE        = 224
BATCH_SIZE      = 32
EPOCHS_HEAD     = 20
EPOCHS_FINETUNE = 40
LR_HEAD         = 1e-3
LR_FINETUNE     = 1e-5
SEED            = 42

print('Config ready ✅')
print(f'Dataset : {DATASET_ROOT}')
print(f'Output  : {OUTPUT_DIR}')

## Cell 8 — Build Label Map

In [ ]:
def normalize_label(folder_name: str) -> str:
    name = folder_name.strip()
    name = re.sub(r'\s+\d+$', '', name)
    name = name.lower().strip()
    name = re.sub(r'[\s\-]+', '_', name)
    name = re.sub(r'[^a-z0-9_]', '', name)
    return name

all_paths, all_labels = [], []
label_set = set()

for category in ['edable', 'toxic']:
    cat_dir = DATASET_ROOT / category
    if not cat_dir.exists():
        print(f'⚠️  Missing: {cat_dir}')
        continue
    for cls_dir in sorted(cat_dir.iterdir()):
        if not cls_dir.is_dir(): continue
        label = normalize_label(cls_dir.name)
        label_set.add(label)
        for img in cls_dir.glob('*.jpg'):
            all_paths.append(img)
            all_labels.append(label)

label_list    = sorted(label_set)
label_to_idx  = {l: i for i, l in enumerate(label_list)}
label_indices = [label_to_idx[l] for l in all_labels]
NUM_CLASSES   = len(label_list)

print(f'Total images : {len(all_paths)}')
print(f'Total classes: {NUM_CLASSES}')
print(f'Labels:\n  ' + '\n  '.join(label_list))

# Save labels.txt to Drive immediately
(OUTPUT_DIR / 'labels.txt').write_text('\n'.join(label_list))
print(f'\nSaved labels.txt to Drive ✅')

## Cell 9 — Class Distribution

In [ ]:
from collections import Counter
counts = Counter(all_labels)

fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(counts.keys(), counts.values(), color=['steelblue' if k in
    [l for l in label_list if (DATASET_ROOT/'edable'/l).exists()] else 'tomato'
    for k in counts.keys()])
ax.set_xticklabels(counts.keys(), rotation=45, ha='right')
ax.set_title('Images per class  (blue=edible, red=toxic)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=100)
plt.show()
print('Chart saved to Drive ✅')

## Cell 10 — Build tf.data Pipeline

In [ ]:
train_paths, val_paths, train_lbls, val_lbls = train_test_split(
    all_paths, label_indices, test_size=0.30, random_state=SEED, stratify=label_indices
)
val_paths, test_paths, val_lbls, test_lbls = train_test_split(
    val_paths, val_lbls, test_size=0.50, random_state=SEED, stratify=val_lbls
)
print(f'Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}')

def make_dataset(paths, labels, augment=False):
    def parse(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32)
        return img, label

    def augment_fn(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        img = tf.image.random_saturation(img, 0.8, 1.2)
        img = tf.image.random_hue(img, 0.05)
        img = tf.image.resize_with_crop_or_pad(img, IMG_SIZE + 20, IMG_SIZE + 20)
        img = tf.image.random_crop(img, [IMG_SIZE, IMG_SIZE, 3])
        img = tf.clip_by_value(img, 0.0, 255.0)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices(([str(p) for p in paths], labels))
    if augment:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(parse, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_paths, train_lbls, augment=True)
val_ds   = make_dataset(val_paths,   val_lbls,   augment=False)
test_ds  = make_dataset(test_paths,  test_lbls,  augment=False)
print('Datasets ready ✅')

## Cell 11 — Build EfficientNetB0 Model

In [ ]:
base = keras.applications.EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling='avg'
)
base.trainable = False

inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.Dropout(0.4)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()
trainable = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f'\nTrainable params: {trainable:,}')

## Cell 12 — Phase 1: Train Head (Base Frozen)
⏱️ ~10 minutes on T4 GPU

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR_HEAD),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        str(OUTPUT_DIR / 'best_head.keras'),
        save_best_only=True, monitor='val_accuracy', verbose=1
    )
]

print('Phase 1: Training head (base frozen)...')
history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_HEAD, callbacks=callbacks_p1
)
print(f'\nPhase 1 best val accuracy: {max(history1.history["val_accuracy"])*100:.2f}% ✅')

## Cell 13 — Phase 2: Fine-Tune Top Layers
⏱️ ~20 minutes on T4 GPU

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(LR_FINETUNE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-8, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        str(OUTPUT_DIR / 'best_model.keras'),
        save_best_only=True, monitor='val_accuracy', verbose=1
    )
]

print('Phase 2: Fine-tuning top 40 layers...')
history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_HEAD + EPOCHS_FINETUNE,
    initial_epoch=len(history1.history['accuracy']),
    callbacks=callbacks_p2
)
print(f'\nPhase 2 best val accuracy: {max(history2.history["val_accuracy"])*100:.2f}% ✅')

## Cell 14 — Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f'\nTest Accuracy : {test_acc * 100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

y_pred, y_true = [], []
for imgs, lbls in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(lbls.numpy())

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=label_list))

## Cell 15 — Training Curves

In [ ]:
acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']
split_ep = len(history1.history['accuracy'])
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs, acc, 'b', label='Train')
ax1.plot(epochs, val_acc, 'r', label='Val')
ax1.axvline(split_ep, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_ylim(0, 1)

ax2.plot(epochs, loss, 'b', label='Train')
ax2.plot(epochs, val_loss, 'r', label='Val')
ax2.axvline(split_ep, color='gray', linestyle='--')
ax2.set_title('Loss'); ax2.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_plot.png', dpi=120)
plt.show()
print('Plot saved to Drive ✅')

## Cell 16 — Export to TFLite → Saved to Drive

In [ ]:
print('Exporting TFLite model...')

# Float16 quantized (recommended — half size, same accuracy)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

tflite_path = OUTPUT_DIR / 'model.tflite'
tflite_path.write_bytes(tflite_model)
print(f'model.tflite saved to Drive ({len(tflite_model)/1024:.0f} KB) ✅')

# Verify the model
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print(f'Input  shape: {inp["shape"]}  dtype: {inp["dtype"]}')
print(f'Output shape: {out["shape"]}  dtype: {out["dtype"]}')
print(f'Output classes: {out["shape"][1]} = {NUM_CLASSES} classes ✅')

## Cell 17 — Quick Inference Test

In [ ]:
from PIL import Image
import random, glob

def predict_tflite(image_path, top_k=3):
    interp = tf.lite.Interpreter(model_path=str(OUTPUT_DIR / 'model.tflite'))
    interp.allocate_tensors()
    inp_det = interp.get_input_details()[0]
    out_det = interp.get_output_details()[0]

    img = Image.open(image_path).convert('RGB').resize((224, 224))
    arr = np.array(img, dtype=np.float32)
    if inp_det['dtype'] == np.float16:
        arr = arr.astype(np.float16)
    interp.set_tensor(inp_det['index'], np.expand_dims(arr, 0))
    interp.invoke()
    probs = interp.get_tensor(out_det['index'])[0].astype(np.float32)
    top_idx = np.argsort(probs)[::-1][:top_k]
    return [(label_list[i], float(probs[i])) for i in top_idx]

samples = glob.glob('/content/dataset_clean/**/*.jpg', recursive=True)
sample  = random.choice(samples)
print(f'Test: {sample}')

plt.imshow(Image.open(sample).resize((224, 224)))
plt.axis('off'); plt.show()

print('Predictions:')
for label, conf in predict_tflite(sample):
    print(f'  {label:<25} {conf*100:5.1f}%  {"█" * int(conf * 40)}')

## Cell 18 — Download Files from Drive
Your files are already saved in **Google Drive → ForestAI_output/**  
You can also download them directly here:

In [ ]:
from google.colab import files

print('Files saved in Google Drive at: MyDrive/ForestAI_output/')
print()
print('Downloading to your computer...')
files.download(str(OUTPUT_DIR / 'model.tflite'))
files.download(str(OUTPUT_DIR / 'labels.txt'))
files.download(str(OUTPUT_DIR / 'training_plot.png'))

print()
print('✅ Done! Next steps:')
print('  1. Copy model.tflite → forest_ai/assets/model.tflite')
print('  2. Copy labels.txt   → forest_ai/assets/labels.txt')
print('  3. Run: flutter build apk --release')